In [0]:
# For account applications, I will define as the first feature the log of income ro reduce the skewness of the distribution

from pyspark.sql.functions import log1p, when

applications = spark.table("fraud_gold.account_application_fraud")

applications = applications.withColumn(
    "log_income",
    log1p("income")
)

# Now, I will define address stability feature to identify patterns of address changes

applications = applications.withColumn(
    "address_stability",
    applications.current_address_months_count / applications.prev_address_months_count
)

# And finally a binary description of the age, counting 1 if the applicant is under 25 to check typical young fraudsters

applications = applications.withColumn(
    "under_25",
    when(applications.customer_age < 25, 1).otherwise(0)
)

# Now I select the target columns and features for processing and modeling

application_features = applications.select(
    "account_id",
    "log_income",
    "address_stability",
    "under_25",
    "name_email_similarity",
    "days_since_request",
    "payment_type",
    "zip_count_4w",
    "is_fraud"
)    

application_features.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("ml_layer.application_features")